# Comparing OpenTofu and Terraform for AWS provisioning

A hands-on look at what an AWS stack looks like under each tool — the shared HCL syntax, how the AWS provider is wired up, how state moves across, and how both fit the same CI/CD shape. I ran the file-level parts locally; anything needing real AWS credentials is written out for inspection, not applied.

## Purpose

OpenTofu and Terraform both consume HCL and expose the same init / plan / apply / destroy workflow, so a small AWS stack (provider block, an S3 bucket with versioning, variables, outputs) is spelled the same way on both sides. This notebook writes one shared AWS configuration into two scratch directories, checks the two copies stay identical, walks through provider resolution and state migration, and sketches the CI/CD stages both tools share.

This is one way to compare them; the docs also suggest going provider by provider — what follows is the smallest experiment that still covers syntax, providers, state, and the pipeline shape.

## Setup

Both CLIs need to be on PATH for the optional binary checks. Everything else is plain file work in throwaway scratch directories, so nothing here touches real AWS infrastructure.

In [ ]:
# last_verified: 2026-09-18 · OpenTofu/Terraform AWS provisioning comparison
# Setup: one scratch directory per tool; all file work, no cloud calls
import hashlib
import shutil
import subprocess
from pathlib import Path

tf_dir = Path("/tmp/ot-compare-terraform-aws")
ot_dir = Path("/tmp/ot-compare-opentofu-aws")
for d in (tf_dir, ot_dir):
    d.mkdir(parents=True, exist_ok=True)
print(f"terraform dir: {tf_dir}")
print(f"opentofu dir:  {ot_dir}")

In [ ]:
# Confirm both binaries respond (informational — file cells below run either way)
for binary in ("terraform", "tofu"):
    try:
        result = subprocess.run([binary, "version"], capture_output=True, text=True, timeout=30)
        first = result.stdout.strip().splitlines()[0] if result.stdout.strip() else "no output"
    except FileNotFoundError:
        first = "not on PATH — skipping binary checks"
    print(f"{binary}: {first}")

## Step 1 — syntax: one shared AWS stack

For everyday AWS resources the HCL is interchangeable: terraform blocks, variable blocks, provider blocks, resource blocks, and outputs are spelled the same way. The stack below is a bucket with versioning plus the usual region/suffix variables — written once, kept in both directories.

In [ ]:
# One shared AWS configuration, written to both scratch directories.
# No provider version is pinned here on purpose — the point is the shared
# shape (blocks, references, outputs), not any particular release.
shared_main = """
terraform {
  required_providers {
    aws = {
      source = "hashicorp/aws"
    }
  }
}

provider "aws" {
  region = var.aws_region
}

variable "aws_region" {
  type    = string
  default = "us-east-1"
}

variable "bucket_suffix" {
  type    = string
  default = "demo-compare"
}

resource "aws_s3_bucket" "demo" {
  bucket = "demo-compare-${var.bucket_suffix}"
  tags = {
    managed_by = "iac-compare"
  }
}

resource "aws_s3_bucket_versioning" "demo" {
  bucket = aws_s3_bucket.demo.id
  versioning_configuration {
    status = "Enabled"
  }
}

output "bucket_name" {
  value = aws_s3_bucket.demo.bucket
}
"""

for d in (tf_dir, ot_dir):
    with open(d / "main.tf", "w") as f:
        f.write(shared_main)
print("Wrote identical main.tf to both directories")
print(f"lines: {len(shared_main.strip().splitlines())}")

In [ ]:
# The two copies should be byte-identical — syntax is shared, so there is
# nothing to translate between sides for this stack.
def sha(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:12]

tf_sum, ot_sum = sha(tf_dir / "main.tf"), sha(ot_dir / "main.tf")
print(f"terraform main.tf: {tf_sum}")
print(f"opentofu  main.tf: {ot_sum}")
print("identical:", tf_sum == ot_sum)

# Quick sanity check on block balance (crude but catches truncation).
text = (tf_dir / "main.tf").read_text()
print("braces balanced:", text.count("{") == text.count("}"))
for block in ("terraform {", "provider", "aws_s3_bucket", "bucket_name"):
    print(f"has {block}: {block in text}")

### What this shows about syntax

- The identical main.tf is valid input for both binaries with no edits — blocks, var. references, resource-to-resource references, and outputs all carry over.
- The habit I settled on: keep the AWS stack in one shared shape and let the binary choice live outside the configuration (which CLI runs it, which state it points at), not inside the HCL.
- Where I would expect divergence is outside basic syntax — provider releases, registry handling, and state — which is what the next steps cover.

## Step 2 — providers

Provider requirements look the same (required_providers with a source address), but each tool resolves them through its own default registry, so the first init on the other side re-resolves providers and rewrites the lock file. In practice the common AWS provider namespace is mirrored, so the hashicorp/aws source works under both — worth verifying per provider rather than assuming, especially for community or private ones.

In [ ]:
# Both directories declare the same provider source address — confirm it
# programmatically instead of eyeballing it.
import re

def provider_sources(path):
    text = Path(path).read_text()
    return re.findall(r'source\s*=\s*"([^"]+)"', text)

tf_srcs = provider_sources(tf_dir / "main.tf")
ot_srcs = provider_sources(ot_dir / "main.tf")
print(f"terraform sources: {tf_srcs}")
print(f"opentofu  sources: {ot_srcs}")
print("same requirements:", tf_srcs == ot_srcs)
print()
print("Expectation when switching sides: the other binary re-resolves on its")
print("first init and refreshes the lock file — plan output after that should")
print("be empty against the same state (checked in step 3).")

## Step 3 — state migration

Both tools default to a local terraform.tfstate JSON file in the working directory, and both configure remote state with a backend block inside the terraform block using the same shape. The snippet below is written out for inspection only — it is not applied, since it needs a real bucket. The migration itself is the copy-and-replan route: copy the directory, init with the new binary, and expect an empty plan before cutting over.

In [ ]:
# Backend configuration shape is identical for both tools (shown, not applied).
# Swap in real bucket/key values before using it for anything real.
backend_example = """
terraform {
  backend "s3" {
    bucket = "example-state-bucket"
    key    = "demo/terraform.tfstate"
    region = "us-east-1"
  }
}
"""

with open(tf_dir / "backend.example.tf", "w") as f:
    f.write(backend_example)
print(backend_example)
print("backend s3 block present:", "backend" in backend_example and "s3" in backend_example)

In [ ]:
# Rehearse the copy half of copy-and-replan with plain file operations:
# duplicate one side's directory and confirm nothing changed in transit.
migrated = Path("/tmp/ot-compare-migrated-aws")
if migrated.exists():
    shutil.rmtree(migrated)
shutil.copytree(ot_dir, migrated)

orig = sorted(p.name for p in ot_dir.iterdir())
copy = sorted(p.name for p in migrated.iterdir())
print(f"original files: {orig}")
print(f"migrated files: {copy}")
print("copy complete:", orig == copy)
print()
print("Next in real use: run tofu init in the copy so providers resolve,")
print("then tofu plan and expect no changes before retiring the original.")

### What this shows about state

- Local state lives at the same path (terraform.tfstate) with the same JSON layout, which is what makes the copy-and-replan move possible at all.
- The remote-backend stanza is a copy, not a rewrite — same block, same keys.
- I keep the original directory around until the new side has done a few clean applies; the copy is cheap and the fallback is worth it.

## Step 4 — CI/CD integration

The pipeline shape is identical for both tools: install the binary, init, plan on every change, hold the plan for review, then apply the saved plan. Only the binary name changes — which means one job template covers both, with the tool choice as a parameter rather than a second pipeline.

In [ ]:
# The CI stage sequence differs only in the binary name — build both
# command lists and show the shape is otherwise the same.
def pipeline(binary):
    return [
        [binary, "init"],
        [binary, "plan", "-out=tfplan"],
        # a person reviews the plan output here; apply runs only after approval
        [binary, "apply", "tfplan"],
    ]

tf_pipe = pipeline("terraform")
ot_pipe = pipeline("tofu")
for t, o in zip(tf_pipe, ot_pipe):
    print("terraform: " + " ".join(t))
    print("opentofu:  " + " ".join(o))
    print()

tf_shape = [[c for c in cmd[1:]] for cmd in tf_pipe]
ot_shape = [[c for c in cmd[1:]] for cmd in ot_pipe]
print("same stage shape (binary name aside):", tf_shape == ot_shape)
print("review gate sits between plan and apply on both sides.")

## Head-to-head

| Aspect | Terraform | OpenTofu |
|---|---|---|
| Config language | HCL (terraform, variable, provider, resource, output blocks) | Same HCL — the shared AWS stack applies under both |
| AWS provider wiring | required_providers with a source address plus an aws provider region block | Same block shape; resolves via its own default registry |
| Daily CLI verbs | init / plan / apply / destroy with the same flags | Same verbs and flags for this workflow |
| Lock files | Records its registry hashes on first init | Same mechanism; expect a re-resolve when switching sides |
| Local state | terraform.tfstate JSON in the working dir | Same path, same JSON layout |
| Remote backends | backend block inside the terraform block | Same block shape — a copy, not a rewrite |
| Migration effort | Starting point | Copy dir, init, empty-plan check, cut over |
| CI/CD shape | Init, plan on change, review gate, apply saved plan | Same stages; only the binary name changes |

## Verify

To confirm the comparison on your own machine:

1. Diff the two main.tf files — expect no differences for this stack.
2. With AWS credentials configured, run terraform init plus terraform plan in one directory and tofu init plus tofu plan in the other — both should propose the same bucket plus versioning resources.
3. For the migration path, copy one side's directory, run tofu init then tofu plan in the copy — expect an empty plan before cutting over.
4. In CI, run the plan stage on a scratch change and confirm the plan output reads the same under either binary.

## What I'd try next

Run the same comparison against a real remote backend with state locking held, and with a community provider, to see where the symmetry breaks — the local file-level experiment above is the easy case, and the interesting differences will show up once locking and third-party providers are involved.

In [ ]:
# Clean up the scratch runs (local-only dirs created by this notebook)
import shutil as _shutil

for _d in (tf_dir, ot_dir, Path("/tmp/ot-compare-migrated-aws")):
    _shutil.rmtree(_d, ignore_errors=True)
print("scratch dirs removed")